# Especies por clase en España y Portugal (GBIF)

Este cuaderno se apoya en el pipeline multi-especie de [`gbif_multiespecie_es_pt.ipynb`](./gbif_multiespecie_es_pt.ipynb): mismos países (`ES`, `PT`), mismos años y la misma carpeta de salida. No descarga el lote de ocurrencias ni pinta el mapa comparativo.

Sirve para **ver las especies** antes de elegir cuáles bajar. Un ranking único de `/v1/occurrence/search` se llena de aves. Aquí se piden cinco especies con coordenadas de cada clase, aunque tengan menos registros:

- Mammalia
- Squamata (lagartos y serpientes; en GBIF no salen como clase Reptilia)
- Amphibia
- Insecta
- Magnoliopsida y Liliopsida
- Aves

La misma lista la muestra la interfaz, en el botón **Especies con coordenadas**.

El kernel del cuaderno es **Python (biodiversidad)**, el intérprete `.venv\Scripts\python.exe`. Si una celda se queda en `[*]` y no imprime el intérprete, interrumpe la ejecución y vuelve a lanzarla con ese kernel. Si una clase no responde, se omite y la celda termina con el resto.

In [1]:
import sys
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(RAIZ / "src") not in sys.path:
    sys.path.insert(0, str(RAIZ / "src"))

from ingesta_gbif_sqlite import (
    CLASES_A_MOSTRAR,
    ESPECIES_POR_CLASE,
    listar_especies_con_coordenadas,
)

PAISES = ["ES", "PT"]
YEAR_RANGE = "2000,2026"
CARPETA_SALIDA = RAIZ / "salidas_gbif_multiespecie"
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)

print("Intérprete:", sys.executable, flush=True)
print("Clases:", ", ".join(CLASES_A_MOSTRAR), flush=True)
print(f"Especies por clase: {ESPECIES_POR_CLASE}", flush=True)
print(f"Salida: {CARPETA_SALIDA}", flush=True)

Intérprete: e:\Users\gracobjo\Documents\proyectosMios\biodiversidad\.venv\Scripts\python.exe
Clases: Mammalia, Squamata, Amphibia, Insecta, Magnoliopsida, Liliopsida, Aves
Especies por clase: 5
Salida: e:\Users\gracobjo\Documents\proyectosMios\biodiversidad\salidas_gbif_multiespecie


## 1. Catálogo

Endpoints:

- `GET /v1/species/match?rank=CLASS` resuelve la clave de cada clase.
- `GET /v1/occurrence/search?classKey={clave}&facet=speciesKey&hasCoordinate=true` devuelve las especies con coordenadas de esa clase.
- `GET /v1/species/{speciesKey}` aporta el nombre, la clase y la familia.

In [2]:
import time

def progreso(mensaje: str) -> None:
    print(mensaje, flush=True)

inicio = time.perf_counter()
filas = listar_especies_con_coordenadas(
    paises=PAISES,
    year_range=YEAR_RANGE,
    limite=ESPECIES_POR_CLASE,
    on_progreso=progreso,
)
print(
    f"Catálogo listo: {len(filas)} especies en {time.perf_counter() - inicio:.0f} s.",
    flush=True,
)
df_especies = pd.DataFrame(filas)
df_especies

Clase 1/7: Mammalia
Clase 2/7: Squamata
Clase 3/7: Amphibia
Clase 4/7: Insecta
Clase 5/7: Magnoliopsida
Clase 6/7: Liliopsida
Clase 7/7: Aves
Nombres de 35 especies
Catálogo listo: 35 especies en 2 s.


,nombre_cientifico,clase,familia,taxon_key,ocurrencias_con_coordenadas,endpoint
0,Sus scrofa,Mammalia,Suidae,7705930,631190,https://api.gbif.org/v1/species/7705930
1,Tarentola mauritanica,Squamata,Phyllodactylidae,2445034,36537,https://api.gbif.org/v1/species/2445034
2,Pelophylax perezi,Amphibia,Ranidae,2426658,43977,https://api.gbif.org/v1/species/2426658
3,Popillia japonica,Insecta,Scarabaeidae,4425774,96319,https://api.gbif.org/v1/species/4425774
4,Quercus suber,Magnoliopsida,Fagaceae,2879411,199743,https://api.gbif.org/v1/species/2879411
5,Arundo donax,Liliopsida,Poaceae,2703041,33863,https://api.gbif.org/v1/species/2703041
6,Passer domesticus,Aves,Passeridae,5231190,1663718,https://api.gbif.org/v1/species/5231190
7,Vulpes vulpes,Mammalia,Canidae,5219243,485885,https://api.gbif.org/v1/species/5219243
8,Psammodromus algirus,Squamata,Lacertidae,2469763,32720,https://api.gbif.org/v1/species/2469763
9,Epidalea calamita,Amphibia,Bufonidae,2422507,23925,https://api.gbif.org/v1/species/2422507


## 2. Ver las clases y las familias

Cada bloque es una clase. La familia permite distinguir, dentro de los mamíferos, cánidos, suidos o cérvidos, y dentro de las aves, el grupo de cada especie.

In [3]:
resumen_clases = (
    df_especies.groupby("clase", sort=False)
    .agg(
        especies=("nombre_cientifico", "count"),
        ocurrencias_con_coordenadas=("ocurrencias_con_coordenadas", "sum"),
    )
    .reset_index()
)
display(resumen_clases)

for clase, bloque in df_especies.groupby("clase", sort=False):
    print(f"\n{clase}")
    display(
        bloque[
            [
                "familia",
                "nombre_cientifico",
                "taxon_key",
                "ocurrencias_con_coordenadas",
            ]
        ].reset_index(drop=True)
    )

,clase,especies,ocurrencias_con_coordenadas
0,Mammalia,5,1790120
1,Squamata,5,118033
2,Amphibia,5,114744
3,Insecta,5,314003
4,Magnoliopsida,5,579987
5,Liliopsida,5,125117
6,Aves,5,7308660



Mammalia


,familia,nombre_cientifico,taxon_key,ocurrencias_con_coordenadas
0,Suidae,Sus scrofa,7705930,631190
1,Canidae,Vulpes vulpes,5219243,485885
2,Cervidae,Capreolus capreolus,5220126,332608
3,Cervidae,Cervus elaphus,2440958,246110
4,Bovidae,Capra pyrenaica,2441054,94327



Squamata


,familia,nombre_cientifico,taxon_key,ocurrencias_con_coordenadas
0,Phyllodactylidae,Tarentola mauritanica,2445034,36537
1,Lacertidae,Psammodromus algirus,2469763,32720
2,Lacertidae,Teira dugesii,2468700,18714
3,Lacertidae,Timon lepidus,2468542,15953
4,Psammophiidae,Malpolon monspessulanus,8796529,14109



Amphibia


... (5 outputs left out)


Liliopsida


,familia,nombre_cientifico,taxon_key,ocurrencias_con_coordenadas
0,Poaceae,Arundo donax,2703041,33863
1,Smilacaceae,Smilax aspera,5295044,24220
2,Asparagaceae,Ruscus aculeatus,2769766,23334
3,Poaceae,Dactylis glomerata,2705308,22699
4,Asparagaceae,Asparagus acutifolius,2768995,21001



Aves


,familia,nombre_cientifico,taxon_key,ocurrencias_con_coordenadas
0,Passeridae,Passer domesticus,5231190,1663718
1,Turdidae,Turdus merula,2490719,1625179
2,Hirundinidae,Hirundo rustica,9515886,1365921
3,Laridae,Larus fuscus,2481174,1349281
4,Sylviidae,Sylvia atricapilla,2492956,1304561


## 3. Guardar el catálogo y elegir especies para el cuaderno multi-especie

El CSV queda en `salidas_gbif_multiespecie/especies_por_clase_es_pt.csv`. La lista de nombres se puede copiar en `ESPECIES_OBJETIVO` de `gbif_multiespecie_es_pt.ipynb` para descargar ocurrencias y mapas. Conviene no copiar las 35 a la vez: ese cuaderno baja hasta 600 registros por país y por especie.

In [4]:
ruta_csv = CARPETA_SALIDA / "especies_por_clase_es_pt.csv"
df_especies.to_csv(ruta_csv, index=False, encoding="utf-8-sig")
print(ruta_csv)

print("\nESPECIES_OBJETIVO = [")
for nombre in df_especies["nombre_cientifico"]:
    print(f'    "{nombre}",')
print("]")

e:\Users\gracobjo\Documents\proyectosMios\biodiversidad\salidas_gbif_multiespecie\especies_por_clase_es_pt.csv

ESPECIES_OBJETIVO = [
    "Sus scrofa",
    "Tarentola mauritanica",
    "Pelophylax perezi",
    "Popillia japonica",
    "Quercus suber",
    "Arundo donax",
    "Passer domesticus",
    "Vulpes vulpes",
    "Psammodromus algirus",
    "Epidalea calamita",
    "Thaumetopoea pityocampa",
    "Olea europaea",
    "Smilax aspera",
    "Turdus merula",
    "Capreolus capreolus",
    "Teira dugesii",
    "Salamandra salamandra",
    "Pararge aegeria",
    "Quercus rotundifolia",
    "Ruscus aculeatus",
    "Hirundo rustica",
    "Cervus elaphus",
    "Timon lepidus",
    "Bufo spinosus",
    "Pieris rapae",
    "Pistacia lentiscus",
    "Dactylis glomerata",
    "Larus fuscus",
    "Capra pyrenaica",
    "Malpolon monspessulanus",
    "Alytes obstetricans",
    "Colias croceus",
    "Vitis vinifera",
    "Asparagus acutifolius",
    "Sylvia atricapilla",
]
